# Recomendaciones por segmento — corrida diaria

Para cada entidad (cliente, sucursal, vendedor...) busca qué ítems (producto, submarca, servicio...)
debería estar comprando y no compra, comparándola **contra sus pares del mismo segmento**. Reemplaza
completa la tabla destino.

Toda la configuración está en **`rec_oracle.py`** y todo el cálculo en **`rec_engine.py`**; este
notebook no tiene nada configurable salvo la celda de parámetros.

```
 ORIGEN (lector)                        MOTOR                          DESTINO (escritor)
 SQL_FUENTE  [:desde, :hasta)  ──>  matriz entidad x ítem  ──>  TABLA_DESTINO (recomendaciones)
 (afinidad + backtest)              batería de algoritmos    ──>  TABLA_DIAGNOSTICO (backtest)
                                    + 3 tipos de recomendación
```

### Nodo en el pipeline (Elyra / OpenShift AI)

| Propiedad | Valor |
|---|---|
| **File Dependencies** | `rec_oracle.py`, `rec_engine.py` |
| **Environment Variables** | `ORA_USER`, `ORA_PASSWORD`, `ORA_DSN`, `ORA_DEST_USER`, `ORA_DEST_PASSWORD`, `ORA_DEST_DSN` |
| **Opcionales** | `RC_FECHA_EJECUCION`, `RC_SELECCION`, `RC_DRY_RUN` |
| **Instalación** | `pip install pandas numpy scipy scikit-learn oracledb` |

### Los tres tipos de recomendación

| Tipo | Qué es | Cómo se estima el USD en juego |
|---|---|---|
| **CRUZADA** | no lo compra y sus pares sí | lo que gasta un par que sí lo compra, ajustado por el tamaño de la entidad |
| **REPOSICION** | lo compraba con cierto ritmo y se atrasó | lo que dejó de comprar desde que se atrasó |
| **BRECHA** | lo compra, pero mucho menos que sus pares | lo que le falta para igualar la participación de sus pares |

In [ ]:
# Parámetros (tag `parameters`)
import os

FECHA_EJECUCION = os.getenv("RC_FECHA_EJECUCION") or None   # "2026-09-15" para simular otro día
SELECCION = os.getenv("RC_SELECCION") or None               # backtest | rrf | ponderado | un algoritmo
DRY_RUN = os.getenv("RC_DRY_RUN", "0") == "1"               # "1" = calcula y no escribe
print(f"FECHA_EJECUCION={FECHA_EJECUCION!r} SELECCION={SELECCION!r} DRY_RUN={DRY_RUN}")

## 1. Setup

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())

import pandas as pd
import rec_oracle as io
from rec_engine import RecEngine

log = io.configurar_logging("recomendaciones")
cfg = io.build_config(fecha_ejecucion=FECHA_EJECUCION, seleccion=SELECCION)
motor = RecEngine(cfg)
desde, hasta = io.ventana(cfg)
log.info("ejecución %s | ventana de lectura %s a %s | %d algoritmos | selección %s",
         motor.fechas.hoy.date(), desde.date(), motor.fechas.ayer.date(), len(cfg.algoritmos), cfg.seleccion)
t_inicio = time.time()

## 2. Tablas destino

Imprime el `CREATE TABLE` de las dos tablas (sin constraints, con la descripción de cada columna) y
valida contra el diccionario de datos que no falte ninguna, **antes** de leer nada.

In [ ]:
print(io.ddl_sugerido(cfg))

with io.conexion_destino() as conn:
    io.validar_tabla(conn, cfg)

## 3. Fuente

Lee `DIAS_AFINIDAD` días para armar la matriz, más `DIAS_BACKTEST` días extra que se reservan para
medir qué algoritmo acierta más. Lo que no quieras recomendar (discontinuados, fletes, notas de
crédito) se filtra en `SQL_FUENTE`.

In [ ]:
with io.conexion_origen() as conn:
    fuente = io.leer_fuente(conn, cfg)
fuente.head(3)

## 4. Cálculo

1. Arma la matriz entidad x ítem de la ventana de afinidad.
2. Asigna a cada entidad el nivel de segmentación **más fino que tenga suficientes entidades**.
3. Si `SELECCION = "backtest"`, entrena con datos hasta hace `DIAS_BACKTEST` días y mide qué se
   compró después: gana el algoritmo que más acertó **en ese segmento**.
4. Con el algoritmo ganador arma las CRUZADAS, y con reglas propias las REPOSICIONES y las BRECHAS.
5. Ordena por USD en juego y se queda con las mejores `MAX_ITEMS_RECO` por entidad.

In [ ]:
out = motor.run(fuente)
out = io.anotar(out, cfg)
diagnostico = io.preparar_diagnostico(motor.diagnostico, cfg)

## 5. Control

In [ ]:
io.resumen(motor, out)
out.head(5).T

In [ ]:
# Backtest: qué tan bien anduvo cada algoritmo en cada segmento. `elegido` marca el que se usó.
motor.diagnostico

## 6. Guardar

Con `MODO_CARGA = "delete"` borra e inserta las dos tablas en una sola transacción: si algo falla,
quedan como estaban.

In [ ]:
if DRY_RUN:
    log.warning("DRY_RUN: no se escribe nada en %s", io.TABLA_DESTINO)
else:
    with io.conexion_destino() as conn:
        io.guardar(conn, out, cfg, diagnostico)
log.info("corrida OK en %.1fs", time.time() - t_inicio)

## Cómo funciona

### La segmentación por niveles

`SEGMENTOS` es una jerarquía escrita del nivel **más grueso al más fino**, por ejemplo
`["BD_SEGMENTO", "BD_SUBSEGMENTO"]`. Cada entidad se compara primero contra la combinación completa
(`MAYORISTA | NORTE`); si a ese grupo no le llegan `MIN_ENTIDADES_SEGMENTO` entidades con compras, se
suelta el último nivel (`MAYORISTA`), y así hasta `GLOBAL`. La columna `BD_NIVEL_SEGMENTO` dice qué
nivel se usó, así se sabe cuánta confianza darle a cada fila.

El orden importa: los niveles se **combinan**, no se reemplazan. Si se escribiera al revés, el norte
mayorista y el norte minorista quedarían mezclados en un mismo grupo `NORTE`.

Comparar dentro del segmento es lo que evita la recomendación tonta: sin segmentar, a un minorista se
le terminan ofreciendo los ítems que compran los mayoristas, sólo porque mueven más plata.

### Cómo se estima el valor: USD esperados en los próximos 90 días

Los tres tipos se miden en la **misma unidad**, para que el ranking compare lo mismo:

```
MT_USD_POTENCIAL = MT_USD_SI_COMPRA x MT_PROB
```

- **`MT_USD_SI_COMPRA`**: lo que gastaría en el horizonte si la compra ocurre. Sale del ticket del
  ítem y del ritmo con que se compra, no de una proyección abierta del silencio.
- **`MT_PROB`**: la probabilidad de que ocurra. En REPOSICION es la probabilidad de recompra; en
  CRUZADA, la tasa de adopción que **midió el backtest** en ese segmento; en BRECHA es 1, porque ya
  lo compra.

### La evidencia de los pares se le presta al que tiene poca propia

Con dos compras, "compraba cada 20 días" es una casualidad, no un ritmo. Con una, no hay ritmo. Pero
sus pares sí tienen ritmo, y eso se puede usar:

```
intervalo estimado = (n propios x intervalo propio + k x intervalo de los pares) / (n propios + k)
```

Con `PESO_PRIOR_PARES = 3`: con **1** compra el estimado es el de los pares; con **2**, se parte la
diferencia; con **20**, es prácticamente el propio. Lo mismo con el ticket, ajustado por el tamaño del
cliente. El motivo dice de dónde salió cada número:

> compró 1 vez; sus pares lo compran cada 32 días, lleva 99 sin comprar (probabilidad de recompra 47%)

### La probabilidad de recompra sale de la historia, no de una fórmula

Para cada nivel de atraso —1, 1,5, 2, 3, 5, 8, 13 veces su intervalo— se cuenta en toda la historia
cuántos casos llegaron a ese atraso y cuántos de esos **volvieron a comprar dentro del horizonte**.
Los silencios que todavía no completaron el horizonte se excluyen, porque no se sabe el resultado.

En una prueba con tasa de retorno conocida del 50%, la curva estimó **50%**; y cayó a 0% cuando el
atraso llegó a 5 intervalos. Si un ítem tiene menos de `MIN_CASOS_RECUPERACION` casos, se usa la
curva del panel entero.

### Cuánta evidencia se exige, y cuánto se puede prometer

| Perilla | Qué evita |
|---|---|
| `MIN_COMPRAS_REPOSICION` | hablar de "su ritmo" sin ningún intervalo propio (con 1 lo presta el segmento) |
| `MAX_CV_INTERVALO` | tratar como atrasado a quien compra salteado, cuando sí tiene ritmo propio medible |
| `MIN_DIAS_COMPRA_ENTIDAD` | recomendarle algo a un cliente con una o dos compras en el año |
| `TOPE_POTENCIAL_POR_HISTORICO` | prometer más de N veces el propio ritmo de compra de ese ítem |
| `TOPE_POTENCIAL_RELATIVO` | que una recomendación valga más que la compra total del cliente en el mismo lapso |

### Por qué se ordena, y el caso de los muy atrasados

`ORDENAR_POR = "esperado"` pone arriba lo que más rinde por visita, que es lo correcto cuando el
vendedor tiene tiempo para tres llamados. Consecuencia a tener presente: un cliente **muy** atrasado
(diez veces su intervalo) tiene probabilidad de recompra casi nula, así que cae debajo de las cruzadas
y puede no entrar en el top.

Si querés armar una campaña de recuperación, esos son justamente los que buscás: poné
`ORDENAR_POR = "bruto"` para ordenar por tamaño de oportunidad sin descontar la probabilidad, o dale
`PISO_PROB = 0.05` para que no valgan cero. También podés filtrar en SQL por `MT_USD_SI_COMPRA` alto y
`MT_PROB` bajo: esa es exactamente la lista de recuperación.

### Una fila por cliente e ítem

Un mismo ítem puede ser a la vez "dejó de comprarlo" y "compra menos que sus pares". Queda **una sola
fila**, con el tipo que mejor lo explica, para que la lista no repita.

### Canasta o repertorio### Canasta o repertorio

`AFINIDAD = "canasta"` mide qué se compra **junto**: mismo cliente, mismo día. Si el cliente compra
una vez por día, el día **es** el ticket, así que no hace falta el número de documento (si lo tenés,
poné `COL_DOCUMENTO` y la canasta pasa a ser el documento). `"repertorio"` mira todo lo que compró en
la ventana, junto o no.

La diferencia no es chica: en la prueba, con la afinidad por canasta el complemento del mismo día se
ofreció primero al **79%** de los clientes; con repertorio, al **0%**, porque ahí los dos candidatos
parecen igual de relacionados.

### El tamaño del cliente

`CORTES_TAMANO = [0.5, 0.8, 0.95]` agrega un nivel de segmentación por cuantiles de compra, así un
cliente mediano no se compara contra uno que compra veinte veces más. Sin esto, lo que el grande
"compra de todo" aparece como oportunidad para el mediano, que nunca lo va a comprar.

### Tipo de documento

Si la fuente distingue facturas, notas de crédito, fletes o ajustes, se declara cuáles son venta,
cuáles restan y cuáles se ignoran. Y si un cliente compró y devolvió todo, no cuenta como comprador
de ese ítem: `EXCLUIR_NETOS_NO_POSITIVOS` se encarga. Si las devoluciones ya vienen netas de BI, no
hace falta configurar nada.

### La batería

| Algoritmo | Qué mira | Cuándo gana |
|---|---|---|
| `popularidad` | qué compran todos los pares | es la línea de base: si nada le gana, no vale la pena personalizar |
| `coseno_item` | con qué se compra junto cada ítem | canastas con productos complementarios |
| `coseno_entidad` | qué compran las entidades más parecidas | cuando hay perfiles de compra claros |
| `svd` | patrones latentes de toda la matriz | matrices densas y con mucha señal |
| `kmeans_valor` | cómo reparte cada entidad su plata entre ítems | cuando el nivel de gasto define el perfil, no la unidad |
| `reglas` | confianza P(B \| A) con lift mayor a 1 | cuando hay asociaciones fuertes y puntuales |

Las cinco últimas personalizan; la primera no. `kmeans_valor` es el que trabaja sobre participación en
USD, para no sumar unidades que no se pueden sumar.

También se pueden combinar todas: `RC_SELECCION=rrf` las fusiona por posición, y `ponderado` por
puntaje normalizado con los pesos de `PESOS`.

### El backtest

Es la parte que evita el autoengaño. Entrena con la matriz hasta hace `DIAS_BACKTEST` días y mira qué
ítems nuevos compró después cada entidad. Las CRUZADAS son las únicas verificables: el ítem no estaba
y apareció.

- **`MT_PRECISION`**: de todo lo recomendado, qué fracción se compró.
- **`MT_RECALL`**: de todo lo que se compró nuevo, qué fracción habíamos recomendado.
- **`MT_USD_ACERTADO`**: cuánta plata había en lo acertado.

Gana el mejor **por segmento**, no uno para todos. Si `popularidad` gana en un segmento, es una señal
útil: ahí no hay nada personalizable y conviene ofrecer lo más vendido.

Un segmento con menos de `min_adopciones_backtest` adopciones en el tramo evaluado no tiene con qué
medirse: usa el ganador del panel entero y lo dice en `BD_MOTIVO_SELECCION`, en vez de elegir con
números que son ruido.

Con datos simulados donde los clientes de un mismo segmento tienen dos gustos distintos, los
algoritmos personalizados **duplicaron la precisión** de la popularidad y no recomendaron nunca ítems
del gusto ajeno, contra el 90% de las veces que lo hacía la popularidad.

### Por qué ordena por plata y no por puntaje

El puntaje dice "le pega"; el USD en juego dice "cuánto vale". El vendedor tiene tiempo para tres
llamados, no para diez: `MT_RANKING` 1 es la de mayor `MT_USD_POTENCIAL`.

## Catálogo de columnas

### REC_CLIENTE_ITEM

| Columna | Descripción |
|---|---|
| `SK_CLIENTE` | Entidad a la que se recomienda (SK_CLIENTE). |
| `BD_CLIENTE` | Entidad a la que se recomienda (BD_CLIENTE). |
| `BK_SUBMARCA` | Ítem recomendado (BK_SUBMARCA). |
| `BD_SUBMARCA` | Ítem recomendado (BD_SUBMARCA). |
| `MT_RANKING` | Orden de la recomendación dentro de la entidad: 1 es la de mayor USD en juego. |
| `BD_TIPO` | CRUZADA (no lo compra y sus pares sí), REPOSICION (lo compraba y se atrasó) o BRECHA (lo compra mucho menos que sus pares). |
| `MT_USD_POTENCIAL` | USD esperados en el horizonte configurado (por defecto 90 días). Es MT_USD_SI_COMPRA por MT_PROB, y es la columna por la que se ordena: los tres tipos quedan en la misma unidad y se pueden comparar. |
| `MT_USD_SI_COMPRA` | USD del horizonte si la compra efectivamente ocurre, sin multiplicar por la probabilidad. Sirve para ver el tamaño de la oportunidad. |
| `MT_PROB` | Probabilidad de que ocurra. REPOSICION: que vuelva a comprar el ítem, según la distribución de intervalos del segmento. CRUZADA: tasa de adopción que midió el backtest en ese segmento. BRECHA: 1, porque ya lo compra. |
| `MT_INTERVALO_ESPERADO` | Días entre compras estimados para este cliente y este ítem, mezclando su propia historia con la de sus pares del segmento: con una compra manda el segmento, con veinte manda él. |
| `MT_COMPRAS_ESPERADAS` | Compras esperadas en el horizonte, según ese intervalo. |
| `MT_MARGEN_POTENCIAL` | MT_USD_POTENCIAL por el margen porcentual del ítem en el segmento. USD. |
| `MT_PUNTAJE` | Puntaje del algoritmo o de la regla. Comparable dentro del mismo tipo y segmento. |
| `MT_PENETRACION_SEGMENTO` | Fracción de las entidades del segmento que compran el ítem. |
| `MT_SOPORTE_SEGMENTO` | Cantidad de entidades del segmento que compran el ítem. |
| `MT_USD_MEDIO_PAR` | USD que gasta en el ítem una entidad del segmento que sí lo compra. USD. |
| `MT_USD_ENTIDAD` | Compra total de la entidad en la ventana de afinidad. USD. |
| `MT_DIAS_SIN_COMPRAR` | Sólo REPOSICION: días desde la última compra del ítem. |
| `MT_DIAS_COMPRA_ITEM` | Días en que la entidad compró este ítem en la ventana. Es la evidencia detrás de REPOSICION y BRECHA: con 2 compras el ritmo es una casualidad, no un ritmo. |
| `MT_INTERVALO_TIPICO` | Días promedio entre compras del ítem por parte de la entidad. |
| `MT_DIAS_COMPRA_ENTIDAD` | Días en que la entidad compró algo en la ventana. Mide cuánta historia sostiene la recomendación. |
| `BD_SEGMENTO` | Segmento contra el que se comparó a la entidad. |
| `BD_NIVEL_SEGMENTO` | Nivel de segmentación usado, o GLOBAL si ninguno tenía suficientes entidades. |
| `BD_ALGORITMO` | Algoritmo que produjo la recomendación, o la regla (regla_reposicion / regla_brecha). |
| `BD_MOTIVO` | Por qué se recomienda, en palabras. |
| `FECHA_CORTE` | Último día incluido en el cálculo (el día anterior a la ejecución). |
| `HUELLA_CONFIG` | Resumen de la configuración con la que se generó la fila (entidad, ítem, segmentos, algoritmos, SQL de la fuente). |
| `FECHA_CARGA` | Fecha y hora en que se cargó la fila. |

### REC_DIAGNOSTICO

| Columna | Descripción |
|---|---|
| `BD_SEGMENTO` | Segmento evaluado. |
| `BD_NIVEL_SEGMENTO` | Nivel de segmentación usado, o GLOBAL. |
| `BD_ALGORITMO` | Algoritmo medido. |
| `MT_ENTIDADES` | Entidades del segmento. |
| `MT_ITEMS_CANDIDATOS` | Ítems que pasaron los mínimos de soporte y penetración. |
| `MT_RECOMENDADOS` | Recomendaciones cruzadas que produjo en el backtest. |
| `MT_ACIERTOS` | Cuántas de esas se compraron después. |
| `MT_PRECISION` | Aciertos sobre recomendados. |
| `MT_ADOPCIONES` | Ítems nuevos que el segmento compró en el tramo evaluado. |
| `MT_ENTIDADES_QUE_ADOPTAN` | Entidades que compraron al menos un ítem nuevo. |
| `MT_RECALL` | Aciertos sobre adopciones. |
| `MT_USD_ACERTADO` | USD comprados en el tramo evaluado de los ítems acertados. |
| `MT_SEGUNDOS` | Lo que tardó el algoritmo en ese segmento. |
| `BD_ELEGIDO` | SI si es el algoritmo que se usó en la corrida final. |
| `BD_MOTIVO_SELECCION` | Por qué se eligió: el que más acertó en el segmento, o el ganador del panel cuando el segmento no tenía adopciones suficientes. |
| `FECHA_CORTE` | Último día incluido en el cálculo. |
| `HUELLA_CONFIG` | Resumen de la configuración. |
| `FECHA_CARGA` | Fecha y hora en que se cargó la fila. |

### Algoritmos disponibles

| Nombre | Qué hace |
|---|---|
| `popularidad` | Penetración del ítem en el segmento. Es la línea de base: ofrecer lo que más compran los pares, sin personalizar. |
| `coseno_item` | Afinidad ítem-ítem por coseno sobre quién compra qué. El puntaje suma la afinidad entre el ítem candidato y lo que la entidad ya compra. |
| `coseno_entidad` | Vecinos parecidos: coseno entre entidades por lo que compran. El puntaje pesa a los k pares más parecidos que sí compran el ítem. |
| `svd` | Factores latentes (SVD truncada) sobre la matriz de compras: patrones de consumo que no se ven mirando ítem por ítem. |
| `kmeans_valor` | k-means sobre cómo reparte cada entidad su dinero entre los ítems (participación en USD, no unidades). El puntaje es la penetración del ítem dentro del grupo. |
| `reglas` | Reglas de asociación: confianza P(compra el candidato | compra lo que ya tiene), exigiendo lift mayor a 1. El puntaje es la mejor regla que le aplica. |